In [1]:
import torch
import torch.nn as nn
from models.utils import get_autoencoder_model, get_classification_model
import config
import numpy as np
from data_preproc.data_preproc_functions import Logger
logger=Logger()

In [7]:
config.filters = [64, 128, 256, 512]
config.strides = [[2]*3, [2]*3, [2]*3, [2]*3]
config.resnet_model_depth = 50

model = get_classification_model(config, 3, 96, 96, 96, 0, None, logger, False)

c:\Users\d.c.macrae\.conda\envs\HNC_310\lib\site-packages\torch\nn\modules\lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
2024-06-20 14:07:37 - INFO: Weight init name: kaiming_uniform.
2024-06-20 14:07:37 - INFO: Using our own weights init scheme for resnet_lrelu.


Layer (type (var_name))                                 Input Shape               Output Shape              Param #                   Kernel Shape
MultiTox_Classifier (MultiTox_Classifier)               [8, 3, 96, 96, 96]        [8, 1]                    --                        --
├─ResNet_LReLU (encoder)                                [8, 3, 96, 96, 96]        [8, 2048, 1, 1, 1]        --                        --
│    └─Conv3d (conv1)                                   [8, 3, 96, 96, 96]        [8, 64, 48, 48, 48]       65,856                    [7, 7, 7]
│    └─BatchNorm3d (bn1)                                [8, 64, 48, 48, 48]       [8, 64, 48, 48, 48]       128                       --
│    └─LeakyReLU (act)                                  [8, 64, 48, 48, 48]       [8, 64, 48, 48, 48]       --                        --
│    └─MaxPool3d (maxpool)                              [8, 64, 48, 48, 48]       [8, 64, 24, 24, 24]       --                        3
│    └─ModuleList (backbo

In [116]:
config.batch_size = 2
autoencoder_model = get_autoencoder_model(config, 1, 96, 96, 96, 0, logger, False)
autoencoder_model2 = get_autoencoder_model(config, 1, 96, 96, 96, 0, logger, False)

Layer (type (var_name))                            Input Shape               Output Shape              Param #                   Kernel Shape
MultiTox_Autoencoder (MultiTox_Autoencoder)        [2, 1, 96, 96, 96]        [2, 1, 96, 96, 96]        --                        --
├─ResNet_LReLU (encoder)                           [2, 1, 96, 96, 96]        [2, 128, 3, 3, 3]         --                        --
│    └─Conv3d (conv1)                              [2, 1, 96, 96, 96]        [2, 16, 48, 48, 48]       5,488                     [7, 7, 7]
│    └─BatchNorm3d (bn1)                           [2, 16, 48, 48, 48]       [2, 16, 48, 48, 48]       32                        --
│    └─LeakyReLU (act)                             [2, 16, 48, 48, 48]       [2, 16, 48, 48, 48]       --                        --
│    └─MaxPool3d (maxpool)                         [2, 16, 48, 48, 48]       [2, 16, 24, 24, 24]       --                        3
│    └─ModuleList (backbone)                       --       

In [117]:
pretrained_path = "C:/Users/d.c.macrae/Documents/DL_NTCP_Multitox/experiments/resnet_test_autoencoder/best_model.pth"
pretrain_dict = torch.load(pretrained_path, map_location=config.device)
model_dict = autoencoder_model.state_dict()
pretrain_dict = {k: v for k, v in pretrain_dict.items() if ((k in model_dict.keys()) and ('output_head' not in k and 'decoder' not in k))}

autoencoder_model.load_state_dict(pretrain_dict, strict=False)

_IncompatibleKeys(missing_keys=['decoder.decoder.0.weight', 'decoder.decoder.0.bias', 'decoder.decoder.2.weight', 'decoder.decoder.2.bias', 'decoder.decoder.4.weight', 'decoder.decoder.4.bias', 'decoder.decoder.6.weight', 'decoder.decoder.6.bias', 'decoder.decoder.8.weight', 'decoder.decoder.8.bias', 'decoder.decoder.10.weight', 'decoder.decoder.10.bias'], unexpected_keys=[])

In [118]:
config.batch_size = 2
classifier_model, _ = get_classification_model(config, 3, 96, 96, 96, 20, None, logger, False)
classifier_model2, _ = get_classification_model(config, 3, 96, 96, 96, 20, None, logger, False)

c:\Users\d.c.macrae\.conda\envs\HNC_310\lib\site-packages\torch\nn\modules\lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '
2024-06-17 10:19:18 - INFO: Weight init name: kaiming_uniform.
2024-06-17 10:19:18 - INFO: Using our own weights init scheme for resnet_lrelu.
2024-06-17 10:19:18 - INFO: Weight init name: kaiming_uniform.
2024-06-17 10:19:18 - INFO: Using our own weights init scheme for resnet_lrelu.


Layer (type (var_name))                                      Input Shape               Output Shape              Param #                   Kernel Shape
MultiTox_Classifier (MultiTox_Classifier)                    [2, 3, 96, 96, 96]        [2, 1]                    --                        --
├─ResNet_LReLU (encoder)                                     [2, 3, 96, 96, 96]        [2, 128, 3, 3, 3]         --                        --
│    └─Conv3d (conv1)                                        [2, 3, 96, 96, 96]        [2, 16, 48, 48, 48]       16,464                    [7, 7, 7]
│    └─BatchNorm3d (bn1)                                     [2, 16, 48, 48, 48]       [2, 16, 48, 48, 48]       32                        --
│    └─LeakyReLU (act)                                       [2, 16, 48, 48, 48]       [2, 16, 48, 48, 48]       --                        --
│    └─MaxPool3d (maxpool)                                   [2, 16, 48, 48, 48]       [2, 16, 24, 24, 24]       --                

In [125]:
autoencoder_model = autoencoder_model.to(config.device)
autoencoder_model2 = autoencoder_model2.to(config.device)
import copy
classifier_model2 = copy.deepcopy(classifier_model)
classifier_model3 = copy.deepcopy(classifier_model)
#classifier_model = classifier_model.to(config.device)
#classifier_model2 = classifier_model2.to(config.device)

In [120]:
model = nn.Linear(1,2)
model2 = copy.deepcopy(model)

for p1, p2 in zip(classifier_model.parameters(), classifier_model2.parameters()):
    print(torch.equal(p1, p2))

True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [123]:
#for key in pretrain_dict.keys():
#    print(key)

with torch.no_grad():
    for key, param_target in classifier_model.named_parameters():
        
        if key in pretrain_dict.keys():
            print(key)

            param_source = pretrain_dict[key]
            print(param_target.shape, param_source.shape)
            
            param_target.data.copy_(param_source.data)

encoder.conv1.weight
torch.Size([16, 3, 7, 7, 7]) torch.Size([16, 1, 7, 7, 7])
encoder.bn1.weight
torch.Size([16]) torch.Size([16])
encoder.bn1.bias
torch.Size([16]) torch.Size([16])
encoder.backbone.block0.0.conv1.weight
torch.Size([16, 16, 1, 1, 1]) torch.Size([16, 16, 1, 1, 1])
encoder.backbone.block0.0.bn1.weight
torch.Size([16]) torch.Size([16])
encoder.backbone.block0.0.bn1.bias
torch.Size([16]) torch.Size([16])
encoder.backbone.block0.0.conv2.weight
torch.Size([16, 16, 3, 3, 3]) torch.Size([16, 16, 3, 3, 3])
encoder.backbone.block0.0.bn2.weight
torch.Size([16]) torch.Size([16])
encoder.backbone.block0.0.bn2.bias
torch.Size([16]) torch.Size([16])
encoder.backbone.block0.0.conv3.weight
torch.Size([64, 16, 1, 1, 1]) torch.Size([64, 16, 1, 1, 1])
encoder.backbone.block0.0.bn3.weight
torch.Size([64]) torch.Size([64])
encoder.backbone.block0.0.bn3.bias
torch.Size([64]) torch.Size([64])
encoder.backbone.block0.0.downsample.0.weight
torch.Size([64, 16, 1, 1, 1]) torch.Size([64, 16, 1, 1

In [124]:
for p1, p2 in zip(classifier_model.parameters(), classifier_model2.parameters()):
    print(torch.equal(p1, p2))

False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
True
True
True
True
True
True
True
True
True
T

In [112]:
for key, param_target in classifier_model.named_parameters():
        
        if key in pretrain_dict.keys():
            print(key)

            #param_source = pretrain_dict[key]
            print(param_target.shape)
            
            #param_target.data.copy_(param_source.data)

encoder.conv1.weight
torch.Size([16, 3, 7, 7, 7])
encoder.bn1.weight
torch.Size([16])
encoder.bn1.bias
torch.Size([16])
encoder.backbone.block0.0.conv1.weight
torch.Size([16, 16, 1, 1, 1])
encoder.backbone.block0.0.bn1.weight
torch.Size([16])
encoder.backbone.block0.0.bn1.bias
torch.Size([16])
encoder.backbone.block0.0.conv2.weight
torch.Size([16, 16, 3, 3, 3])
encoder.backbone.block0.0.bn2.weight
torch.Size([16])
encoder.backbone.block0.0.bn2.bias
torch.Size([16])
encoder.backbone.block0.0.conv3.weight
torch.Size([64, 16, 1, 1, 1])
encoder.backbone.block0.0.bn3.weight
torch.Size([64])
encoder.backbone.block0.0.bn3.bias
torch.Size([64])
encoder.backbone.block0.0.downsample.0.weight
torch.Size([64, 16, 1, 1, 1])
encoder.backbone.block0.0.downsample.1.weight
torch.Size([64])
encoder.backbone.block0.0.downsample.1.bias
torch.Size([64])
encoder.backbone.block0.1.conv1.weight
torch.Size([16, 64, 1, 1, 1])
encoder.backbone.block0.1.bn1.weight
torch.Size([16])
encoder.backbone.block0.1.bn1.bi

In [115]:
with torch.no_grad():
    for (name1, param1), (name2, param2) in zip(classifier_model.encoder.named_parameters(), autoencoder_model.encoder.named_parameters()):
        

        #print(param1.size() == param2.size())

        #print(param1[0] == param1[1])
        size1 = param1.size()
        size2 = param2.size()

        

        if size1 == size2:
            #print("hi")
            print(param1==param2)
            #param1 = param2
            #print("easy")
        # else:
        #     if np.sum(size1 != size2) == 1 and size1[1] != size2[1]:
        #         print(name1, name2)
        #         for i in range(size1[1]):
        #             param1[:,i] = param2[:,0]
                    
        #         print("yes")


tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True], device='cuda:0')
tensor([True, True, True, True, True, True, True, True, True, True, True, True,
        True, True, True, True], device='cuda:0')
tensor([[[[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]]],



        [[[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]],


         [[[True]]]],





In [81]:
for p1, p2 in zip(classifier_model.parameters(), classifier_model2.parameters()):
    print(torch.equal(p1, p2))

False
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True
True


In [61]:
for (name1, param1), (name2, param2) in zip(classifier_model.encoder.named_parameters(), autoencoder_model.encoder.named_parameters()):
    print((param1 == param2).all())
    print(name1, name2)

    #print(param1.size() == param2.size())


tensor(True, device='cuda:0')
conv1.weight conv1.weight
tensor(True, device='cuda:0')
bn1.weight bn1.weight
tensor(True, device='cuda:0')
bn1.bias bn1.bias
tensor(False, device='cuda:0')
backbone.block0.0.conv1.weight backbone.block0.0.conv1.weight
tensor(True, device='cuda:0')
backbone.block0.0.bn1.weight backbone.block0.0.bn1.weight
tensor(True, device='cuda:0')
backbone.block0.0.bn1.bias backbone.block0.0.bn1.bias
tensor(False, device='cuda:0')
backbone.block0.0.conv2.weight backbone.block0.0.conv2.weight
tensor(True, device='cuda:0')
backbone.block0.0.bn2.weight backbone.block0.0.bn2.weight
tensor(True, device='cuda:0')
backbone.block0.0.bn2.bias backbone.block0.0.bn2.bias
tensor(False, device='cuda:0')
backbone.block0.0.conv3.weight backbone.block0.0.conv3.weight
tensor(True, device='cuda:0')
backbone.block0.0.bn3.weight backbone.block0.0.bn3.weight
tensor(True, device='cuda:0')
backbone.block0.0.bn3.bias backbone.block0.0.bn3.bias
tensor(False, device='cuda:0')
backbone.block0.0.

In [23]:
param1.shape

torch.Size([128])

In [21]:
size1 == size2

True

In [32]:
size1 = [16,1,7,7,7]
size2 = [16,3,7,7,7]
import numpy as np

# check if only the last dimension is different
if np.sum(size1 != size2) == 1 and size1[1] != size2[1]:
    print("yes")


yes


In [13]:
for name, param in autoencoder_model.encoder.named_parameters():
    print(name)

    if hasattr(autoencoder_model.encoder, name):
        print("yes")

conv1.weight
bn1.weight
bn1.bias
backbone.block0.0.conv1.weight
backbone.block0.0.bn1.weight
backbone.block0.0.bn1.bias
backbone.block0.0.conv2.weight
backbone.block0.0.bn2.weight
backbone.block0.0.bn2.bias
backbone.block0.0.conv3.weight
backbone.block0.0.bn3.weight
backbone.block0.0.bn3.bias
backbone.block0.0.downsample.0.weight
backbone.block0.0.downsample.1.weight
backbone.block0.0.downsample.1.bias
backbone.block0.1.conv1.weight
backbone.block0.1.bn1.weight
backbone.block0.1.bn1.bias
backbone.block0.1.conv2.weight
backbone.block0.1.bn2.weight
backbone.block0.1.bn2.bias
backbone.block0.1.conv3.weight
backbone.block0.1.bn3.weight
backbone.block0.1.bn3.bias
backbone.block0.2.conv1.weight
backbone.block0.2.bn1.weight
backbone.block0.2.bn1.bias
backbone.block0.2.conv2.weight
backbone.block0.2.bn2.weight
backbone.block0.2.bn2.bias
backbone.block0.2.conv3.weight
backbone.block0.2.bn3.weight
backbone.block0.2.bn3.bias
backbone.block1.0.conv1.weight
backbone.block1.0.bn1.weight
backbone.blo